In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("silver_ais_events")
print(f"Silver rows: {silver_df.count()}")
silver_df.printSchema()
display(silver_df.limit(10))

In [0]:
gold_df = (
    silver_df
    .groupBy("event_date")
    .agg(
        F.count("*").alias("total_events"),
        F.countDistinct("vessel_id").alias("unique_vessels"),
        F.round(F.avg("speed_knots"), 2).alias("avg_speed_knots"),
        F.sum(F.when(F.col("is_moving") == True, 1).otherwise(0)).alias("moving_events"),
        F.sum(F.when(F.col("speed_category") == "stationary", 1).otherwise(0)).alias("stationary_events"),
        F.sum(F.when(F.col("speed_category") == "fast", 1).otherwise(0)).alias("fast_events")
    )
)

In [0]:
display(gold_df.orderBy("event_date"))

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_traffic_daily")

In [0]:
spark.table("gold_traffic_daily").printSchema()
display(spark.table("gold_traffic_daily"))